# Module 3 — Sequential Multi-Channel Fraud Detection
**Phase 1 (feature pipeline) + Phase 2 (Hybrid Bi-LSTM model)**

Run top to bottom on Google Colab. Set **Runtime → Change runtime type → GPU** first.

This notebook expects two files in the working directory:
`phase1_features.py` and `phase2_model.py` (upload them in step 1).

## Step 1 — Upload the two .py files
Run this, then pick `phase1_features.py` and `phase2_model.py` from your computer.

In [ ]:
from google.colab import files
up = files.upload()   # select phase1_features.py and phase2_model.py
print(sorted(up.keys()))

## Step 2 — Get the Sparkov dataset
**Option A (recommended): Kaggle API.** Upload your `kaggle.json` (Kaggle → Account → Create New API Token).

In [ ]:
# Option A: Kaggle download
from google.colab import files
print('Upload your kaggle.json'); files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip -q install kaggle
!kaggle datasets download -d kartik2112/fraud-detection -p . --unzip
import glob; print(glob.glob('*.csv'))

In [ ]:
# Option B: if you'd rather upload the CSVs manually, run THIS instead of Option A
# from google.colab import files
# files.upload()   # select fraudTrain.csv and fraudTest.csv
#
# The files must end up named exactly fraudTrain.csv and fraudTest.csv
# in the working directory.

## Step 3 — Install dependencies
Colab already has tensorflow, pandas, numpy, scikit-learn. We add the ONNX exporters and SHAP.

In [ ]:
!pip -q install tf2onnx onnx shap

## Step 4 — Sanity check Phase 1 (verify learnability before modelling)
This is the diagnostic that distinguishes a usable dataset from a random-label one.
A learnable dataset shows clear fraud-vs-legit separation.

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv('fraudTrain.csv', nrows=300000)
df['ts'] = pd.to_datetime(df['trans_date_trans_time']); df['hour'] = df.ts.dt.hour
print('fraud rate     :', round(df.is_fraud.mean(), 4))
print('mean amount    : legit', round(df[df.is_fraud==0].amt.mean(),2),
      '| fraud', round(df[df.is_fraud==1].amt.mean(),2))
print('night fraud (22-3h) rate:', round(df[df.hour.isin([22,23,0,1,2,3])].is_fraud.mean(),4))
print('day   fraud (4-21h) rate:', round(df[~df.hour.isin([22,23,0,1,2,3])].is_fraud.mean(),4))

## Step 5 — Build features (Phase 1) and run the model + ablation (Phase 2)
On a GPU runtime this runs on the FULL dataset. Pass `nrows=` to cap for a quick test.

In [ ]:
from phase2_model import run_ablation
# Full dataset: nrows=None.  Quick test: nrows=200000
results = run_ablation('fraudTrain.csv', 'fraudTest.csv', nrows=None)

## Step 5b — Full evaluation with figures (PR curve, ROC, confusion matrix)
Run after Step 5. Produces the metrics summary and four figures inline,
and saves them + the trained model. This is your evaluation display.

In [ ]:
"""
Module 3 - Phase 2 Evaluation (interim presentation)
====================================================
Trains the full Bi-LSTM + network model, evaluates it on the temporal test
split, and writes presentation-ready figures + a metrics summary.

Outputs:
  eval_metrics.txt        - text summary you can read off in the talk
  fig_pr_curve.png        - precision-recall curve (the headline figure)
  fig_roc_curve.png       - ROC curve
  fig_confusion.png       - confusion matrix at the chosen threshold
  fig_threshold.png       - precision/recall/F1 vs threshold

Run:
    python3 evaluate_model.py             # default 300k rows (fast, representative)
    python3 evaluate_model.py 0           # full dataset
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_recall_curve, roc_curve,
                             confusion_matrix, precision_score,
                             recall_score, f1_score)

from phase1_features import build_dataset, FEATURE_CONFIG
from phase2_model import build_model, binary_focal_loss
from tensorflow import keras

NROWS = None   # None = full dataset; set e.g. 300000 for a faster run

TRAIN = "fraudTrain.csv"
TEST = "fraudTest.csv"

# ---- build features ----
train = build_dataset(TRAIN, nrows=NROWS)
test = build_dataset(TEST, fit_from=train, nrows=NROWS)
cfg = FEATURE_CONFIG
Xs, Xn, y = train["X_seq"], train["X_net"], train["y"]
cut = int(len(y) * 0.85)

# ---- train full bidirectional model ----
model = build_model(window=cfg["window"], n_seq_features=cfg["n_seq_features"],
                    n_categories=train["n_categories"],
                    n_net_features=len(cfg["net_features"]), bidirectional=True)
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss=binary_focal_loss(gamma=2.0, alpha=0.25))
es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=2,
                                   restore_best_weights=True)
model.fit([Xs[:cut], Xn[:cut]], y[:cut],
          validation_data=([Xs[cut:], Xn[cut:]], y[cut:]),
          epochs=12, batch_size=2048, callbacks=[es], verbose=2)

# ---- predict ----
va = model.predict([Xs[cut:], Xn[cut:]], batch_size=8192, verbose=0).ravel()
yv = y[cut:]
te = model.predict([test["X_seq"], test["X_net"]], batch_size=8192, verbose=0).ravel()
yt = test["y"]

# ---- choose threshold on validation, NOT test ----
p, r, thr = precision_recall_curve(yv, va)
f1v = 2 * p * r / (p + r + 1e-9)
best = thr[max(0, np.argmax(f1v) - 1)]

pred = (te >= best).astype(int)
roc = roc_auc_score(yt, te)
prauc = average_precision_score(yt, te)
P = precision_score(yt, pred, zero_division=0)
R = recall_score(yt, pred, zero_division=0)
F = f1_score(yt, pred, zero_division=0)
cm = confusion_matrix(yt, pred)

# ---- text summary ----
lines = [
    "MODULE 3 - EVALUATION SUMMARY",
    "=" * 40,
    f"Train rows used : {len(y):,}   (val slice {len(yv):,})",
    f"Test rows       : {len(yt):,}",
    f"Test fraud rate : {yt.mean():.4f}  ({int(yt.sum())} fraud / {len(yt):,})",
    "",
    "PRIMARY METRICS (imbalanced -> PR-AUC is the headline)",
    f"  PR-AUC (average precision) : {prauc:.4f}",
    f"  ROC-AUC                    : {roc:.4f}",
    "",
    f"AT OPERATING THRESHOLD = {best:.3f}  (chosen on validation set)",
    f"  Precision : {P:.3f}",
    f"  Recall    : {R:.3f}",
    f"  F1        : {F:.3f}",
    "",
    "CONFUSION MATRIX  [[TN, FP], [FN, TP]]",
    f"  {cm.tolist()}",
    f"  True negatives : {cm[0,0]:,}",
    f"  False positives: {cm[0,1]:,}   (legit flagged as fraud)",
    f"  False negatives: {cm[1,0]:,}   (fraud missed)",
    f"  True positives : {cm[1,1]:,}   (fraud caught)",
    "",
    "NOTE: accuracy is omitted on purpose - at <1% fraud a model that",
    "predicts 'never fraud' scores >99% accuracy, so accuracy is misleading.",
]
summary = "\n".join(lines)
open("eval_metrics.txt", "w").write(summary)
print(summary)

# ---- figures ----
def style(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.25)

# PR curve
pr_p, pr_r, _ = precision_recall_curve(yt, te)
fig, ax = plt.subplots(figsize=(5.5, 4.2))
ax.plot(pr_r, pr_p, color="#534AB7", lw=2)
ax.axhline(yt.mean(), color="#999", ls="--", lw=1,
           label=f"baseline (random) = {yt.mean():.3f}")
ax.scatter([R], [P], color="#D85A30", zorder=5,
           label=f"operating point\nP={P:.2f} R={R:.2f}")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title(f"Precision-Recall curve  (PR-AUC = {prauc:.3f})")
ax.legend(fontsize=9); style(ax); fig.tight_layout()
fig.savefig("fig_pr_curve.png", dpi=150); plt.close(fig)

# ROC curve
fpr, tpr, _ = roc_curve(yt, te)
fig, ax = plt.subplots(figsize=(5.5, 4.2))
ax.plot(fpr, tpr, color="#1D9E75", lw=2)
ax.plot([0, 1], [0, 1], color="#999", ls="--", lw=1, label="random")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title(f"ROC curve  (ROC-AUC = {roc:.3f})")
ax.legend(fontsize=9); style(ax); fig.tight_layout()
fig.savefig("fig_roc_curve.png", dpi=150); plt.close(fig)

# Confusion matrix
fig, ax = plt.subplots(figsize=(4.6, 4.2))
im = ax.imshow(cm, cmap="Purples")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["pred legit", "pred fraud"])
ax.set_yticklabels(["true legit", "true fraud"])
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, f"{v:,}", ha="center", va="center",
            color="white" if v > cm.max() / 2 else "#333", fontsize=12)
ax.set_title(f"Confusion matrix @ thr={best:.2f}")
fig.tight_layout()
fig.savefig("fig_confusion.png", dpi=150); plt.close(fig)

# Threshold sweep
fig, ax = plt.subplots(figsize=(5.5, 4.2))
ax.plot(thr, p[:-1], label="precision", color="#185FA5", lw=1.8)
ax.plot(thr, r[:-1], label="recall", color="#D85A30", lw=1.8)
ax.plot(thr, f1v[:-1], label="F1", color="#534AB7", lw=1.8)
ax.axvline(best, color="#999", ls="--", lw=1, label=f"chosen thr={best:.2f}")
ax.set_xlabel("Threshold"); ax.set_ylabel("Score")
ax.set_title("Precision / Recall / F1 vs threshold (validation)")
ax.legend(fontsize=9); style(ax); fig.tight_layout()
fig.savefig("fig_threshold.png", dpi=150); plt.close(fig)

model.save("module3_model.keras")
print("\nSaved: eval_metrics.txt, fig_pr_curve.png, fig_roc_curve.png, "
      "fig_confusion.png, fig_threshold.png, module3_model.keras")


# Show the figures inline in the notebook
from IPython.display import Image, display
for fn in ["fig_pr_curve.png", "fig_roc_curve.png", "fig_confusion.png", "fig_threshold.png"]:
    display(Image(fn))


## Step 6 — Phase 4: SHAP explainability on the trained model
Explains which features and which timestep drove a flagged transaction.
Uses KernelExplainer on a small background sample (fast and model-agnostic).

In [ ]:
import numpy as np, shap, tensorflow as tf
from tensorflow import keras
from phase1_features import build_dataset, FEATURE_CONFIG, SEQ_NUMERIC, NET_FEATURES
from phase2_model import binary_focal_loss

model = keras.models.load_model('module3_model.keras',
                                custom_objects={'loss': binary_focal_loss()})
test = build_dataset('fraudTest.csv', fit_from=build_dataset('fraudTrain.csv', nrows=200000),
                     nrows=50000)
Xs, Xn, y = test['X_seq'], test['X_net'], test['y']
W = FEATURE_CONFIG['window']; F = FEATURE_CONFIG['n_seq_features']

# flatten the two inputs into one vector so SHAP can perturb them
def flat(xs, xn): return np.concatenate([xs.reshape(len(xs), -1), xn], axis=1)
def unflat(v):
    n = len(v); seq = v[:, :W*F].reshape(n, W, F); net = v[:, W*F:]
    return [seq.astype('float32'), net.astype('float32')]
f = lambda v: model.predict(unflat(v), verbose=0).ravel()

bg = flat(Xs[:100], Xn[:100])
# pick a few fraud cases the model caught
prob = model.predict([Xs, Xn], batch_size=8192, verbose=0).ravel()
idx = np.where((y == 1) & (prob > 0.5))[0][:5]
expl = shap.KernelExplainer(f, bg)
sv = expl.shap_values(flat(Xs[idx], Xn[idx]), nsamples=200)

feat_names = [f'{c}@t-{W-1-s}' for s in range(W) for c in (['cat']+SEQ_NUMERIC)] + NET_FEATURES
for i, row in enumerate(np.array(sv)[0] if isinstance(sv, list) else sv):
    top = np.argsort(np.abs(row))[::-1][:5]
    print(f'\nFraud case {idx[i]} (p={prob[idx[i]]:.2f}) top drivers:')
    for t in top: print(f'   {feat_names[t]:24s} {row[t]:+.3f}')

## Step 7 — Download the trained artifacts

In [ ]:
from google.colab import files
for fn in ['module3_model.keras', 'module3_model.onnx']:
    try: files.download(fn)
    except Exception as e: print('skip', fn, e)